In [ ]:
import os
import time
import pyotp
import pandas as pd
from dotenv import load_dotenv
from SmartApi import SmartConnect
from datetime import datetime, timedelta

In [2]:
load_dotenv()

True

In [3]:
API_KEY = os.getenv("ANGEL_API_KEY")
CLIENT_ID = os.getenv("ANGEL_CLIENT_ID")
PIN = os.getenv("ANGEL_PIN")
TOTP_TOKEN = os.getenv("ANGEL_TOTP_TOKEN")

In [4]:
url = "https://margincalculator.angelone.in/OpenAPI_File/files/OpenAPIScripMaster.json"
instruments = pd.read_json(url)

companies = instruments.loc[
    instruments["exch_seg"].isin(["NSE", "nse_cm"])
    & instruments["symbol"].str.endswith("-EQ", na=False),
    ["name", "symbol", "token"]
].sort_values("name").reset_index(drop=True)

companies["token"] = companies["token"].astype(str)
companies.to_csv("../data/nse_companies.csv", index=False)

companies

,name,symbol,token
0,011NSETEST,011NSETEST-EQ,14747
1,021NSETEST,021NSETEST-EQ,14751
2,031NSETEST,031NSETEST-EQ,14753
3,041NSETEST,041NSETEST-EQ,14755
4,051NSETEST,051NSETEST-EQ,14758
...,...,...,...
2720,ZSARACOM,ZSARACOM-EQ,762985
2721,ZUARI,ZUARI-EQ,29050
2722,ZUARIIND,ZUARIIND-EQ,3827
2723,ZYDUSLIFE,ZYDUSLIFE-EQ,7929


In [5]:
obj = SmartConnect(api_key=API_KEY)

totp = pyotp.TOTP(TOTP_TOKEN).now()
session = obj.generateSession(CLIENT_ID, PIN, totp)

AUTH_TOKEN = session['data']['jwtToken']
FEED_TOKEN = obj.getfeedToken()
print("Session active")

[I 261011 02:06:57 smartConnect:124] in pool


Session active


In [6]:
symbols = ["SBIN-EQ", "LICI-EQ", "HAL-EQ", "NTPC-EQ", "ONGC-EQ"]
selected = companies.set_index("symbol").loc[symbols]

start_year = 2025
end_year = 2026

for symbol, company in selected.iterrows():

    ROOT = f"../data/{symbol.replace('-EQ', '')}"
    token = str(company["token"])

    for year in range(start_year, end_year + 1):

        max_month = 1 if year == 2026 else 12

        for month in range(1, max_month + 1):

            month_start = datetime(year, month, 1)
            next_month = (
                datetime(year + 1, 1, 1)
                if month == 12
                else datetime(year, month + 1, 1)
            )
            month_end = next_month - timedelta(days=1)

            params = {
                "exchange": "NSE",
                "symboltoken": token,
                "interval": "ONE_DAY",
                "fromdate": month_start.strftime("%Y-%m-%d 00:00"),
                "todate": month_end.strftime("%Y-%m-%d 23:59")
            }

            try:
                res = obj.getCandleData(params)

                if res.get("status") is not True:
                    print(f"{symbol} {year}-{month:02d}: {res.get('message')}")
                    continue

                if not res.get("data"):
                    print(f"{symbol} {year}-{month:02d}: No data")
                    continue

                df = pd.DataFrame(
                    res["data"],
                    columns=["datetime", "open", "high", "low", "close", "volume"]
                )
                df = df.drop_duplicates("datetime").sort_values("datetime")

                os.makedirs(f"{ROOT}/{year}", exist_ok=True)
                file_path = f"{ROOT}/{year}/{month:02d}.csv"
                df.to_csv(file_path, index=False)

                print(f"Saved {file_path} ({len(df)} rows)")

            except Exception as e:
                print(f"{symbol} {year}-{month:02d}: {type(e).__name__}")

            finally:
                time.sleep(1)

Saved ../data/SBIN/2025/01.csv (23 rows)
Saved ../data/SBIN/2025/02.csv (20 rows)
Saved ../data/SBIN/2025/03.csv (19 rows)
Saved ../data/SBIN/2025/04.csv (19 rows)
Saved ../data/SBIN/2025/05.csv (21 rows)
Saved ../data/SBIN/2025/06.csv (21 rows)
Saved ../data/SBIN/2025/07.csv (23 rows)
Saved ../data/SBIN/2025/08.csv (19 rows)
Saved ../data/SBIN/2025/09.csv (22 rows)
Saved ../data/SBIN/2025/10.csv (21 rows)
Saved ../data/SBIN/2025/11.csv (19 rows)
Saved ../data/SBIN/2025/12.csv (22 rows)
Saved ../data/SBIN/2026/01.csv (20 rows)
Saved ../data/LICI/2025/01.csv (23 rows)
Saved ../data/LICI/2025/02.csv (20 rows)
Saved ../data/LICI/2025/03.csv (19 rows)
Saved ../data/LICI/2025/04.csv (19 rows)
Saved ../data/LICI/2025/05.csv (21 rows)
Saved ../data/LICI/2025/06.csv (21 rows)
Saved ../data/LICI/2025/07.csv (23 rows)
Saved ../data/LICI/2025/08.csv (19 rows)
Saved ../data/LICI/2025/09.csv (22 rows)
Saved ../data/LICI/2025/10.csv (21 rows)
Saved ../data/LICI/2025/11.csv (19 rows)
Saved ../data/LI